## Tutorial GWAS com a ferramenta Hail

Esse tutorial tem como objetivo principal ensinar o fluxo e comandos básicos para realizar um GWAS utilizando o pacote Hail do Broad Institute

### Instalção de pacotes necessários

In [1]:
! pip install hail

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.6/455.6 MB 57.6 MB/s  0:00:05
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.1/74.1 MB 49.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 118.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import hail as hl

In [3]:
from pprint import pprint

from hail.plot import show

hl.plot.output_notebook()

### Baixando os dados

Os dados utilizados neste tutorial são uma pequena parte do dataset do 1000 Genomes. O arquivo é um VCF de em torno de 20MB. Serão utilizados também metadados.

In [4]:
hl.utils.get_1kg('data/')

Initializing Hail with default parameters...
Running on Apache Spark version 4.1.4
SparkUI available at http://d85111055280:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.140-1b30333664d7
LOGGING: writing to /content/hail-20261121-2245-0.2.140-1b30333664d7.log


O Hail utiliza um formato para processamento e representação dos dados chamado MatrixTable, que foi feito para deixar as análises mais rápidas. Nos comandos abaixo, o arquivo VCF é convertido para esse formato e a MatrixTable **mt** é criada.





In [5]:
hl.import_vcf('data/1kg.vcf.bgz').write('data/1kg.mt', overwrite=True)

### Conhecendo mais das funcionalidades do Hail

In [6]:
mt = hl.read_matrix_table('data/1kg.mt')

A MatrixTable tem uma estrutura um pouco diferente das tabelas convencionais. Abaixo são rodados alguns comandos para se ter acesso a algumas linhas da MatrixTable.

In [7]:
mt.rows().select().show(10)

,
locus,alleles
locus<GRCh37>,array<str>
1:904165,"[""G"",""A""]"
1:909917,"[""G"",""A""]"
1:986963,"[""C"",""T""]"
1:1563691,"[""T"",""G""]"
1:1707740,"[""T"",""G""]"
1:2252970,"[""C"",""T""]"
1:2284195,"[""T"",""C""]"
1:2779043,"[""T"",""C""]"


In [8]:
mt.s.show(5)

""
s
str
"""HG00096"""
"""HG00099"""
"""HG00105"""
"""HG00118"""
"""HG00129"""


In [9]:
mt.entry.take(10)

[Struct(GT=Call(alleles=[0, 0], phased=False), AD=[4, 0], DP=4, GQ=12, PL=[0, 12, 147]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[8, 0], DP=8, GQ=24, PL=[0, 24, 315]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[8, 0], DP=8, GQ=23, PL=[0, 23, 230]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[7, 0], DP=7, GQ=21, PL=[0, 21, 270]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[5, 0], DP=5, GQ=15, PL=[0, 15, 205]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[4, 0], DP=4, GQ=11, PL=[0, 11, 88]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[2, 0], DP=2, GQ=6, PL=[0, 6, 58]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[5, 0], DP=5, GQ=14, PL=[0, 14, 138]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[5, 0], DP=5, GQ=15, PL=[0, 15, 189]),
 Struct(GT=Call(alleles=[0, 0], phased=False), AD=[13, 0], DP=13, GQ=39, PL=[0, 39, 405])]

Anotações comumente são cruciais para estudos genéticos de associação. Elas comumente possuem informações relacionadas ao fenótipo, ancestralidade, sexo e covariáveis. Nos comandos abaixo é demonstrando como utilizar o arquivo **1kg_annotations.txt** para adicionar novas colunas a MatrixTable.

In [10]:
table = hl.import_table('data/1kg_annotations.txt', impute=True).key_by('Sample')

In [11]:
table.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'Sample': str 
    'Population': str 
    'SuperPopulation': str 
    'isFemale': bool 
    'PurpleHair': bool 
    'CaffeineConsumption': int32 
----------------------------------------
Key: ['Sample']
----------------------------------------


In [12]:
table.show(width=100)

,,,,,
Sample,Population,SuperPopulation,isFemale,PurpleHair,CaffeineConsumption
str,str,str,bool,bool,int32
"""HG00096""","""GBR""","""EUR""",False,False,4
"""HG00097""","""GBR""","""EUR""",True,True,4
"""HG00098""","""GBR""","""EUR""",False,False,5
"""HG00099""","""GBR""","""EUR""",True,False,4
"""HG00100""","""GBR""","""EUR""",True,False,5
"""HG00101""","""GBR""","""EUR""",False,True,1
"""HG00102""","""GBR""","""EUR""",True,True,6
"""HG00103""","""GBR""","""EUR""",False,True,5


In [13]:
mt = mt.annotate_cols(pheno=table[mt.s])

In [14]:
mt.col.describe()

--------------------------------------------------------
Type:
        struct {
        s: str, 
        pheno: struct {
            Population: str, 
            SuperPopulation: str, 
            isFemale: bool, 
            PurpleHair: bool, 
            CaffeineConsumption: int32
        }
    }
--------------------------------------------------------
Source:
Index:
    ['column']
--------------------------------------------------------


O Hail possui várias funções para gerar estatísticas simples, porém importantes em relação aos datasets analisados:


*   counter: conta o número de ocorrências de um elemento único
*   stats: produz estatísticas básicas sobre distribuições numéricas



In [15]:
print(table.aggregate(hl.agg.counter(table.SuperPopulation)))

{'AFR': 1018, 'AMR': 535, 'EAS': 617, 'EUR': 669, 'SAS': 661}


In [16]:
pprint(table.aggregate(hl.agg.stats(table.CaffeineConsumption)))

Struct(mean=3.9837142857142855,
       stdev=1.7021055628070711,
       min=-1.0,
       max=10.0,
       n=3500,
       sum=13943.0)


Abaixo, é possível observar que existe uma diferença no tamanho da tabela de metadados e da MatrixTable.

In [17]:
table.count()

3500

In [18]:
mt.count_cols()

284

Podemos utilizar o métodos aggregate_cols para recuperar somente as nossas amostras de interesse.

In [19]:
mt.aggregate_cols(hl.agg.counter(mt.pheno.SuperPopulation))

{'AFR': 76, 'AMR': 34, 'EAS': 72, 'EUR': 47, 'SAS': 55}

In [20]:
pprint(mt.aggregate_cols(hl.agg.stats(mt.pheno.CaffeineConsumption)))

Struct(mean=4.415492957746479,
       stdev=1.577763427465917,
       min=0.0,
       max=9.0,
       n=284,
       sum=1254.0)


In [21]:
snp_counts = mt.aggregate_rows(hl.agg.counter(hl.Struct(ref=mt.alleles[0], alt=mt.alleles[1])))
pprint(snp_counts)

{Struct(ref='G', alt='C'): 111,
 Struct(ref='A', alt='G'): 1929,
 Struct(ref='C', alt='A'): 494,
 Struct(ref='C', alt='G'): 150,
 Struct(ref='T', alt='C'): 1864,
 Struct(ref='T', alt='G'): 466,
 Struct(ref='G', alt='T'): 477,
 Struct(ref='A', alt='T'): 75,
 Struct(ref='T', alt='A'): 77,
 Struct(ref='G', alt='A'): 2367,
 Struct(ref='C', alt='T'): 2418,
 Struct(ref='A', alt='C'): 451}


In [22]:
from collections import Counter

counts = Counter(snp_counts)
counts.most_common()

[(Struct(ref='C', alt='T'), 2418),
 (Struct(ref='G', alt='A'), 2367),
 (Struct(ref='A', alt='G'), 1929),
 (Struct(ref='T', alt='C'), 1864),
 (Struct(ref='C', alt='A'), 494),
 (Struct(ref='G', alt='T'), 477),
 (Struct(ref='T', alt='G'), 466),
 (Struct(ref='A', alt='C'), 451),
 (Struct(ref='C', alt='G'), 150),
 (Struct(ref='G', alt='C'), 111),
 (Struct(ref='T', alt='A'), 77),
 (Struct(ref='A', alt='T'), 75)]

### Controle de qualidade
A partir de agora, iremos investigar algumas colunas com dados relacionados a qualidade do sequenciamento (ou genotipagem) das variantes. Essa etapa pode nos ajudar a identificar amostras ou variantes que precisam ser decartadas para as etapas posteriores.

Abaixo, plotamos um histograma da distribuição da coluna DP. No arquivo VCF, DP significa "Depth", que nada mais é do que a profundidade das reads (total de reads) cobrindo aquela posição genômica.

In [23]:
p = hl.plot.histogram(mt.DP, range=(0, 30), bins=30, title='DP Histogram', legend='DP')
show(p)

O controle de qualidade é muito importante e não há uma carta coringa para todos os datasets. Cada dataset possui suas especificidades. Por isso, é importante redrobar a atenção nessa etapa.

O Hail facilita o calculo de métricas de qualidade com a função **sample_qc**, a qual produz um conjunto de métricas e guarda elas numa coluna.

In [24]:
mt = hl.sample_qc(mt)

In [25]:
mt.col.describe()

--------------------------------------------------------
Type:
        struct {
        s: str, 
        pheno: struct {
            Population: str, 
            SuperPopulation: str, 
            isFemale: bool, 
            PurpleHair: bool, 
            CaffeineConsumption: int32
        }, 
        sample_qc: struct {
            dp_stats: struct {
                mean: float64, 
                stdev: float64, 
                min: float64, 
                max: float64
            }, 
            gq_stats: struct {
                mean: float64, 
                stdev: float64, 
                min: float64, 
                max: float64
            }, 
            call_rate: float64, 
            n_called: int64, 
            n_not_called: int64, 
            n_filtered: int64, 
            n_hom_ref: int64, 
            n_het: int64, 
            n_hom_var: int64, 
            n_non_ref: int64, 
            n_singleton: int64, 
            n_snp: int64, 
            n_insertio

"Call rate" é uma métrica que denota a porcentagem de loci no array para os quais a chamada do genótipo foi feita com sucesso. Se uma amostra possui bom call rate, a amostra possui boa qualidade e genotipagem. Baixas taxas de call rate podem indicar problemas com a amostra ou problemas no processamento.

In [26]:
p = hl.plot.histogram(mt.sample_qc.call_rate, range=(0.88, 1), legend='Call Rate')
show(p)

"GQ" refere-se a qualidade da identificação genótipo. Em arquivos de sequenciamento e genotipagem, é apresentada pelo "Phred quality score". Esse score é logariticamente relacionado a probabilidade de erro da chamada da variante e é definido como:

𝐐 = -10 logP

Dessa forma, se o score de uma base é de 30, as chances de uma base ter sido chamada incorretamente é de 1 em 1000.

Abaixo, é plotado a distribuição da média de GQ por amostra.

In [27]:
p = hl.plot.histogram(mt.sample_qc.gq_stats.mean, range=(10, 70), legend='Mean Sample GQ')
show(p)

Em seguida, podemos plotar um gráfico de dispersão onde procuramos ver a relação da média de cobertura e o call rate de cada amostra.

In [28]:
p = hl.plot.scatter(mt.sample_qc.dp_stats.mean, mt.sample_qc.call_rate, xlabel='Mean DP', ylabel='Call Rate')
show(p)

Dado as análises descritivas vistas anteriormente, podemos fazer filtros.

In [29]:
mt = mt.filter_cols((mt.sample_qc.dp_stats.mean >= 4) & (mt.sample_qc.call_rate >= 0.97))
print('After filter, %d/284 samples remain.' % mt.count_cols())

After filter, 250/284 samples remain.


Em seguida, podemos fazer filtragens baseadas em métricas de qualidade dos genótipos especificamente. Por exemplo, se um genótipo foi chamado como homozigoto para o alelo de referência, mas temos uma proporção de alelos alternativos maior que 10%, provavelmente há um erro.


alelo = 0/0
70:30

No código abaixo, calculamos uma porporção baseada na quantidade de alelos alternativos sobre o número total de contagem de alelos. Em seguida, são feitos filtros que consideram situações "normais".

In [30]:
ab = mt.AD[1] / hl.sum(mt.AD)

filter_condition_ab = (
    (mt.GT.is_hom_ref() & (ab <= 0.1)) # é homozigoto para o referência, e ab é menor ou igual a 10%
    | (mt.GT.is_het() & (ab >= 0.25) & (ab <= 0.75)) # é heterozigoto e ab é maior ou igual a 25% ou menor ou igual a 75% (ou seja, a proporção do alelo alternativo não é muito baixa, nem alta)
    | (mt.GT.is_hom_var() & (ab >= 0.9)) # é homozigoto para o alelo alternativo, e o ab é maior ou igual a 90% (proporções abaixo disso não nos trazem muita confiança para a chamada de homozigoto alternativo)
)

fraction_filtered = mt.aggregate_entries(hl.agg.fraction(~filter_condition_ab))
print(f'Filtering {fraction_filtered * 100:.2f}% entries out of downstream analysis.')
mt = mt.filter_entries(filter_condition_ab)

Filtering 3.60% entries out of downstream analysis.


In [31]:
p = hl.plot.histogram(mt.sample_qc.gq_stats.mean, range=(10, 70), legend='Mean Sample GQ')
show(p)

A função variant_qc produz estatísticas da mesma forma que a função mostrada anteriormente.

In [32]:
mt = hl.variant_qc(mt)

In [33]:
mt.row.describe()

--------------------------------------------------------
Type:
        struct {
        locus: locus<GRCh37>, 
        alleles: array<str>, 
        rsid: str, 
        qual: float64, 
        filters: set<str>, 
        info: struct {
            AC: array<int32>, 
            AF: array<float64>, 
            AN: int32, 
            BaseQRankSum: float64, 
            ClippingRankSum: float64, 
            DP: int32, 
            DS: bool, 
            FS: float64, 
            HaplotypeScore: float64, 
            InbreedingCoeff: float64, 
            MLEAC: array<int32>, 
            MLEAF: array<float64>, 
            MQ: float64, 
            MQ0: int32, 
            MQRankSum: float64, 
            QD: float64, 
            ReadPosRankSum: float64, 
            set: str
        }, 
        variant_qc: struct {
            dp_stats: struct {
                mean: float64, 
                stdev: float64, 
                min: float64, 
                max: float64
            }, 

Abaixo podemos ver versões do mesmos gráficos gerados anteriormente, mas agora para as variantes e não amostras.

In [34]:
p = hl.plot.histogram(mt.variant_qc.dp_stats.mean, range=(4,20), legend='Mean variant DP')
show(p)

In [35]:
p = hl.plot.histogram(mt.variant_qc.gq_stats.mean, range=(10,70), legend='Mean variant GQ')
show(p)

### GWAS

O GWAS, pelo fato de como sua metodologia estatística funciona, é mais adequado para variantes com frequência alélica mais comum na população. Geralmente, são utilizadas variantes de frequência alélica maior que 5%, chamadas de variantes comuns. Nesse tutorial faremos esse corte.

Além disso, utilizaramos também outro corte relacionado a variantes que não estejam longe do equilíbrio de Hardy-Weinberg (o que sugere um erro de sequenciamento)

*Fazer este próximo bloco  primeiro*

A título de comparação, se fizessemos um filtro para variantes com frequência alélica menores que 1%, teríamos pouco mais de 9500 variantes.

In [36]:
mt_rare = mt.filter_rows(mt.variant_qc.AF[1] < 0.01)

mt_rare = mt.filter_rows(mt.variant_qc.p_value_hwe > 1e-6)

print('Samples: %d  Variants: %d' % (mt_rare.count_cols(), mt_rare.count_rows()))

Samples: 250  Variants: 9558


Agora, o filtro "oficial".

In [37]:
mt = mt.filter_rows(mt.variant_qc.AF[1] > 0.01)

mt = mt.filter_rows(mt.variant_qc.p_value_hwe > 1e-6)

print('Samples: %d  Variants: %d' % (mt.count_cols(), mt.count_rows()))

Samples: 250  Variants: 7774


O fenótipo a ser utilizado no GWAS será o consumo de cafeína.

In [38]:
gwas = hl.linear_regression_rows(y=mt.pheno.CaffeineConsumption, x=mt.GT.n_alt_alleles(), covariates=[1.0])
gwas.row.describe()

--------------------------------------------------------
Type:
        struct {
        locus: locus<GRCh37>, 
        alleles: array<str>, 
        n: int32, 
        sum_x: float64, 
        y_transpose_x: float64, 
        beta: float64, 
        standard_error: float64, 
        t_stat: float64, 
        p_value: float64
    }
--------------------------------------------------------
Source:
Index:
    ['row']
--------------------------------------------------------


O gráfico mais utilizado para avaliar os resultados do GWAS é o Manhattan plot. No eixo x desse gráfico, temos a posições (ou coordenadas) genômicas referentes as cada um dos SNPs testados. Cada SNP e seu respectivo beta (tamanho de efeito) possui uma significância estatística, a qual pode ser julgada pelo seu p-valor. A fim de deixar esse p-valor mais "livel", apliacamos um logarítmo. Esse valor é indicado no eixo y.

In [39]:
p = hl.plot.manhattan(gwas.p_value)
show(p)

O Q-Q plot (quantile-quantile plot) é um gráfico de duas dimensões onde vemos a relação entre a distribuição dos p-valores observados em nosso experimento e os p-valores esperados de uma distribuição randomizada. No caso de um GWAS onde não há nenhum SNP com associação significativa, devemos esperar uma linha diagonal (como a linha vermelha no gráfico abaixo). Se nosso GWAS não tem poder estatístico, devemos esperar esse comportamento.

No caso de um GWAS com SNPs significantes, o QQ plot terá uma "cauda". A maioria dos SNPs vai seguir uma dsitribuição uniforme (não são significativos, a hipótese nula é aceita), mas uma pequena porção terá p-valores significantes (baixíssimos, ou seja, com -log(p-valores) altos) e estes são os da "cauda". Esse é o cenário ideal que queremos ver.

No entanto, no gráfico abaixo, não vemos esse comportamento, o que indica que nosso dados podem estar com algum fator que atrapalha o GWAS.




In [40]:
p = hl.plot.qq(gwas.p_value)
show(p)

Um dos fatores que podem impactar o GWAS é a estrutura populacional. A estrutura populacional diz respeito a diferenças nas frequências alélicas entre subpopulações de uma mesma espécie. Teoricamente, numa população que tem cruzamentos ao acaso, espera-se que essas frequências sejam similares entres os grupos. No entanto, esse não é o caso quase sempre e, de fato, existem diferenças na estrutura populacional das subpopulações humanas, devido aos vários processos demográficos, naturais (por exemplo) que ocorreram desde a origem de nossa espécie.

Sendo assim, é de extrema importância levar em consideração esses fatores nos modelos de associação. Uma forma de "obter numericamente" essa estrutura populacional, é aplicando ténicas de redução de dimensionalidade. Uma vez que estamos lidando com vários SNPs desses indivíduos, podemos reduzir essa imensa dimensionalidade para dois, por exemplo, e visualizar a diferença populacional dentro de nossas amostras. O método utilizado aqui é Principal Component Analysis (PCA).

In [41]:
eigenvalues, pcs, _ = hl.hwe_normalized_pca(mt.GT)

In [42]:
pprint(eigenvalues)

[18.08411146784072,
 9.984076405601838,
 3.5406872298059464,
 2.655598108390123,
 1.5968527017244059,
 1.5405241027955252,
 1.5077135041162115,
 1.4744976712480322,
 1.4676905390347434,
 1.4461994473306543]


In [43]:
pcs.show(5, width=100)

,
s,scores
str,array<float64>
"""HG00096""","[-1.22e-01,2.81e-01,1.10e-01,-1.27e-01,6.68e-02,3.29e-03,-2.26e-02,-4.26e-02,-9.30e-02,-1.83e-01]"
"""HG00099""","[-1.14e-01,2.89e-01,1.06e-01,-6.78e-02,4.72e-02,2.87e-02,5.28e-03,1.57e-02,1.75e-02,1.98e-02]"
"""HG00105""","[-1.09e-01,2.79e-01,9.95e-02,-1.06e-01,8.79e-02,1.44e-02,2.80e-02,3.38e-02,-1.08e-03,-2.25e-02]"
"""HG00118""","[-1.26e-01,2.95e-01,7.58e-02,-1.08e-01,1.76e-02,7.91e-03,-5.25e-02,-3.05e-02,2.00e-02,7.78e-02]"
"""HG00129""","[-1.06e-01,2.86e-01,9.69e-02,-1.15e-01,1.03e-02,2.65e-02,-8.51e-02,-2.49e-02,5.67e-02,8.31e-03]"


In [44]:
mt = mt.annotate_cols(scores=pcs[mt.s].scores)

p = hl.plot.scatter(mt.scores[0], mt.scores[1], label=mt.pheno.SuperPopulation, title='PCA', xlabel='PC1', ylabel='PC2')
show(p)

O modelo de GWAS abaixo levará em consideração as covariaves sexo, e os três primeiros componentes principais do PCA.

In [45]:
gwas = hl.linear_regression_rows(
    y=mt.pheno.CaffeineConsumption,
    x=mt.GT.n_alt_alleles(),
    covariates=[1.0, mt.pheno.isFemale, mt.scores[0], mt.scores[1], mt.scores[2]],
)

Agora nosso Q-Q plot parece estar bem melhor!

In [46]:
p = hl.plot.qq(gwas.p_value)
show(p)

O Manhattan plot também mudou de comportamento e agora temos menos variantes com p-valores muito baixos.

In [47]:
p = hl.plot.manhattan(gwas.p_value)
show(p)